# Forecast, scenarios and assumed cost parameters

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT/'data'
RESULTS_DIR = PROJECT_ROOT/'results'
SETTINGS = json.loads((DATA_DIR/'project_settings.json').read_text(encoding='utf-8'))
SEED = SETTINGS['seed']
N_SKUS = SETTINGS['n_skus']
TRAIN_WEEKS = SETTINGS['train_weeks']
HOLDOUT_WEEKS = SETTINGS['holdout_weeks']
HORIZON = SETTINGS['horizon']
N_SCENARIOS = SETTINGS['n_scenarios']
SERVICE_LEVEL = SETTINGS['service_level']
train_weekly = pd.read_csv(DATA_DIR/'train_weekly.csv')
holdout = pd.read_csv(DATA_DIR/'holdout_actual.csv')
assert train_weekly['sku'].nunique()==N_SKUS
print('Loaded',len(train_weekly),'training rows')

Loaded 384 training rows


### Create a simple, transparent forecast using **training data only**

We use a weighted blend of week-four lag and recent eight-week average.

In [2]:
SKUS = tuple(sorted(train_weekly['sku'].unique()))
train_matrix = (train_weekly.pivot(index='week', columns='sku', values='demand')
                .loc[range(TRAIN_WEEKS), list(SKUS)].to_numpy(dtype=float))
if train_matrix.shape != (TRAIN_WEEKS, N_SKUS):
    raise ValueError(f'Unexpected training matrix shape {train_matrix.shape}')
training_predictions = []
training_errors = []
for t in range(8, TRAIN_WEEKS):
    pred = 0.65 * train_matrix[t-4] + 0.35 * train_matrix[t-8:t].mean(axis=0)
    training_predictions.append(pred)
    training_errors.append(train_matrix[t] - pred)
residuals = np.asarray(training_errors)
future_pred = np.asarray([
    0.65 * train_matrix[-4 + (t % 4)] + 0.35 * train_matrix[-8:].mean(axis=0)
    for t in range(HORIZON)
])
forecast = pd.DataFrame([
    {'sku': i, 'week': t, 'forecast': max(0., float(future_pred[t,j])),
     'train_residual_std': float(residuals[:,j].std(ddof=1))}
    for j,i in enumerate(SKUS) for t in range(HORIZON)
])
display(forecast.head(12))

,sku,week,forecast,train_residual_std
0,FOODS_3_064_CA_1_validation,0,190.8625,35.359911
1,FOODS_3_064_CA_1_validation,1,200.6125,35.359911
2,FOODS_3_064_CA_1_validation,2,186.3125,35.359911
3,FOODS_3_064_CA_1_validation,3,164.8625,35.359911
4,FOODS_3_064_CA_1_validation,4,190.8625,35.359911
5,FOODS_3_064_CA_1_validation,5,200.6125,35.359911
6,FOODS_3_090_CA_1_validation,0,370.3500,268.230408
7,FOODS_3_090_CA_1_validation,1,354.1000,268.230408
8,FOODS_3_090_CA_1_validation,2,358.0000,268.230408
9,FOODS_3_090_CA_1_validation,3,358.0000,268.230408


In [3]:
# Evaluating demand forecasting accuracy
forecast_check = forecast.merge(holdout.rename(columns={'demand':'actual_sales'}),on=['sku','week'], validate='one_to_one')
forecast_check['absolute_error'] = (forecast_check['forecast'] - forecast_check['actual_sales']).abs()
forecast_check['squared_error'] = (forecast_check['forecast'] - forecast_check['actual_sales'])**2
print('Holdout MAE:', round(forecast_check['absolute_error'].mean(), 3))
print('Holdout RMSE:', round(np.sqrt(forecast_check['squared_error'].mean()), 3))
display(forecast_check.head(12))

Holdout MAE: 44.976
Holdout RMSE: 74.442


,sku,week,forecast,train_residual_std,actual_sales,absolute_error,squared_error
0,FOODS_3_064_CA_1_validation,0,190.8625,35.359911,164,26.8625,721.593906
1,FOODS_3_064_CA_1_validation,1,200.6125,35.359911,160,40.6125,1649.375156
2,FOODS_3_064_CA_1_validation,2,186.3125,35.359911,153,33.3125,1109.722656
3,FOODS_3_064_CA_1_validation,3,164.8625,35.359911,171,6.1375,37.668906
4,FOODS_3_064_CA_1_validation,4,190.8625,35.359911,182,8.8625,78.543906
5,FOODS_3_064_CA_1_validation,5,200.6125,35.359911,225,24.3875,594.750156
6,FOODS_3_090_CA_1_validation,0,370.3500,268.230408,356,14.3500,205.922500
7,FOODS_3_090_CA_1_validation,1,354.1000,268.230408,427,72.9000,5314.410000
8,FOODS_3_090_CA_1_validation,2,358.0000,268.230408,300,58.0000,3364.000000
9,FOODS_3_090_CA_1_validation,3,358.0000,268.230408,371,13.0000,169.000000


### Construct demand scenarios from training errors

In [4]:
rng_scenarios = np.random.default_rng(SEED + 1)
scenario_rows = []
for k in range(N_SCENARIOS):
    scenario_id = f'TRAIN{k:03d}'
    for t in range(HORIZON):
        residual_vector = residuals[rng_scenarios.integers(0, len(residuals))]
        simulated_demand = np.rint(np.maximum(0., future_pred[t] + residual_vector)).astype(int)
        for j, i in enumerate(SKUS):
            scenario_rows.append({'scenario': scenario_id, 'probability': 1.0/N_SCENARIOS,
                                  'sku': i, 'week': t, 'demand': int(simulated_demand[j])})
train_scenarios = pd.DataFrame(scenario_rows)
print('Scenario rows:', len(train_scenarios), '| unique scenarios:', train_scenarios['scenario'].nunique())
display(train_scenarios.head(12))

Scenario rows: 576 | unique scenarios: 16


,scenario,probability,sku,week,demand
0,TRAIN000,0.0625,FOODS_3_064_CA_1_validation,0,218
1,TRAIN000,0.0625,FOODS_3_090_CA_1_validation,0,361
2,TRAIN000,0.0625,FOODS_3_120_CA_1_validation,0,169
3,TRAIN000,0.0625,FOODS_3_252_CA_1_validation,0,315
4,TRAIN000,0.0625,FOODS_3_586_CA_1_validation,0,294
5,TRAIN000,0.0625,FOODS_3_587_CA_1_validation,0,255
6,TRAIN000,0.0625,FOODS_3_064_CA_1_validation,1,209
7,TRAIN000,0.0625,FOODS_3_090_CA_1_validation,1,1
8,TRAIN000,0.0625,FOODS_3_120_CA_1_validation,1,232
9,TRAIN000,0.0625,FOODS_3_252_CA_1_validation,1,295


## Define **assumed** operational parameters

M5 **does not** give real purchase costs, stockout penalties, warehouse capacity, initial on-hand inventory, or supplier lead times. The values below are explicitly **illustrative costs in arbitrary monetary units**, scaled to each selected SKU's historical sales volume.

In [5]:
p_rows = []
for j,sku in enumerate(SKUS):
    avg = float(train_matrix[:,j].mean())
    max_training_demand = int(train_scenarios.loc[train_scenarios['sku'].eq(sku), 'demand'].max())
    per_unit_cost = [13.,17.,21.,25.,29.,33.][j]
    p_rows.append({
        'sku':sku,
        'initial_inventory': int(round(0.65*avg)),
        'max_order_per_week': int(max(math.ceil(2.0*avg + 20), math.ceil(1.1*max_training_demand + 10))),
        'unit_order_cost': per_unit_cost,
        'fixed_order_cost': 14. + j*4,
        'holding_cost': 0.5 + 0.1*(j%4),
        'stockout_penalty': 105. + j*9,
        'waste_penalty': 1.6*per_unit_cost,
    })
params = pd.DataFrame(p_rows).set_index('sku')
WAREHOUSE_CAPACITY = float(math.ceil(2.5 * train_matrix.mean(axis=0).sum() + 30))
conf = {'service_level': SERVICE_LEVEL, 'warehouse_capacity': WAREHOUSE_CAPACITY,
        'source': 'M5 observed sales + assumed operating parameters',
        'training_weeks': TRAIN_WEEKS, 'holdout_weeks': HOLDOUT_WEEKS,
        'scenario_count': N_SCENARIOS, 'seed': SEED}
print('Warehouse capacity (assumed):', WAREHOUSE_CAPACITY)
display(params)

Warehouse capacity (assumed): 3959.0


,initial_inventory,max_order_per_week,unit_order_cost,fixed_order_cost,holding_cost,stockout_penalty,waste_penalty
sku,,,,,,,
FOODS_3_064_CA_1_validation,122,396,13.0,14.0,0.5,105.0,20.8
FOODS_3_090_CA_1_validation,222,905,17.0,18.0,0.6,114.0,27.2
FOODS_3_120_CA_1_validation,184,633,21.0,22.0,0.7,123.0,33.6
FOODS_3_252_CA_1_validation,177,564,25.0,26.0,0.8,132.0,40.0
FOODS_3_586_CA_1_validation,195,620,29.0,30.0,0.5,141.0,46.4
FOODS_3_587_CA_1_validation,122,396,33.0,34.0,0.6,150.0,52.8


## Conventional forecast-driven **base-stock baseline** 

The baseline is a simple, interpretable **open-loop replenishment rule**. At the start of each week, it estimates an order so the **projected pre-sales stock** covers the forecast plus a safety buffer:

$$q_{it}(z)=\min\bigl(M_i,\max(0,\lceil \widehat D_{it}+z\widehat\sigma_i-\widehat I_{i,t-1}\rceil)\bigr),$$

where $\widehat I$ is **forecast-projected** inventory. 

We try $z\in\{0,0.5,1,1.5,2,2.5,3\}$ **on training scenarios only** and select the first $z$ value when fill-rate reaches 95$%$. If none meets it, select the highest training fill rate and mark that the target was not achieved. 

In [6]:
BASELINE_BUFFER_GRID = (0.,0.5,1.,1.5,2.,2.5,3.)

def make_fixed_base_stock_plan(buffer_multiplier):
    """Generate fixed weekly orders using forecast + safety buffer and projected stock."""
    projected = {i: float(params.loc[i, 'initial_inventory']) for i in SKUS}
    forecast_lookup = {(str(r.sku), int(r.week)): (float(r.forecast), float(r.train_residual_std))
                       for r in forecast.itertuples(index=False)}
    rows = []
    for t in range(HORIZON):
        for i in SKUS:
            predicted, sigma = forecast_lookup[i, t]
            target_before_sales = max(0., predicted + buffer_multiplier * sigma)
            limit = int(params.loc[i, 'max_order_per_week'])
            order = int(np.clip(math.ceil(max(0., target_before_sales - projected[i])), 0, limit))
            rows.append({'sku': i, 'week': t, 'order_qty': order})
            projected[i] = max(0., projected[i] + order - predicted)
    return pd.DataFrame(rows)


def training_baseline_metrics(plan):
    """Forward-simulate a fixed plan on training demand scenarios, with assumed costs.

    The same operational accounting is used later for BOTH the optimized and baseline plans: immediate weekly receipts, 
    lost sales (no backorders), end-week capacity disposal by lowest waste penalty first, and per-week holding costs.
    """
    q = {(str(r.sku), int(r.week)): int(r.order_qty) for r in plan.itertuples(index=False)}
    first_stage_cost = sum(float(params.loc[i, 'unit_order_cost']) * q[i,t]
                           + float(params.loc[i, 'fixed_order_cost']) * (q[i,t] > 0)
                           for i in SKUS for t in range(HORIZON))
    demand_by_path = {(str(r.scenario), str(r.sku), int(r.week)): float(r.demand)
                      for r in train_scenarios.itertuples(index=False)}
    scenario_probabilities = (train_scenarios[['scenario', 'probability']]
                              .drop_duplicates().set_index('scenario')['probability'].to_dict())
    weighted_cost = first_stage_cost
    weighted_loss = weighted_demand = weighted_disposal = 0.
    for scenario, probability in scenario_probabilities.items():
        stock = {i: float(params.loc[i, 'initial_inventory']) for i in SKUS}
        scenario_cost = first_stage_cost
        scenario_loss = scenario_demand = scenario_disposal = 0.
        for t in range(HORIZON):
            for i in SKUS:
                d = demand_by_path[scenario, i, t]
                available = stock[i] + q[i,t]
                sold = min(available, d)
                lost = d - sold
                stock[i] = available - sold
                scenario_loss += lost
                scenario_demand += d
                scenario_cost += lost * float(params.loc[i, 'stockout_penalty'])
            excess = max(0., sum(stock.values()) - float(conf['warehouse_capacity']))
            for i in sorted(SKUS, key=lambda k: float(params.loc[k, 'waste_penalty'])):
                removed = min(stock[i], excess)
                excess -= removed
                stock[i] -= removed
                scenario_disposal += removed
                scenario_cost += removed * float(params.loc[i, 'waste_penalty'])
                if excess < 1e-9:
                    break
            scenario_cost += sum(stock[i] * float(params.loc[i, 'holding_cost']) for i in SKUS)
        weighted_cost += probability * (scenario_cost - first_stage_cost)
        weighted_loss += probability * scenario_loss
        weighted_demand += probability * scenario_demand
        weighted_disposal += probability * scenario_disposal
    return {'train_cost': weighted_cost,
            'train_fill_rate': 1. - weighted_loss / max(1., weighted_demand),
            'train_lost_units': weighted_loss,
            'train_disposed_units': weighted_disposal}

baseline_candidates = []
for z in BASELINE_BUFFER_GRID:
    candidate_orders = make_fixed_base_stock_plan(z)
    baseline_candidates.append({'buffer_multiplier': z,
                                **training_baseline_metrics(candidate_orders)})
baseline_tuning = pd.DataFrame(baseline_candidates)
meets_target = baseline_tuning['train_fill_rate'] >= SERVICE_LEVEL - 1e-9
if meets_target.any():
    eligible = baseline_tuning.loc[meets_target]
    # selected = eligible.sort_values(['train_cost', 'buffer_multiplier']).iloc[0]
    selected = eligible.iloc[0]
    baseline_training_service_met = True
else:
    selected = baseline_tuning.sort_values(['train_fill_rate', 'train_cost'], ascending=[False, True]).iloc[0]
    baseline_training_service_met = False
BASELINE_BUFFER = float(selected['buffer_multiplier'])
baseline_orders = make_fixed_base_stock_plan(BASELINE_BUFFER)
print('Selected forecast/safety-buffer multiplier:', BASELINE_BUFFER)
print('Baseline meets training scenario service target:', baseline_training_service_met)
print('Baseline training fill rate:', f"{float(selected['train_fill_rate']):.2%}")
display(baseline_tuning)
display(baseline_orders.head(12))

Selected forecast/safety-buffer multiplier: 1.0
Baseline meets training scenario service target: True
Baseline training fill rate: 95.88%


,buffer_multiplier,train_cost,train_fill_rate,train_lost_units,train_disposed_units
0,0.0,270177.02500,0.911961,769.9375,0.0
1,0.5,248608.22500,0.937946,542.6875,0.0
2,1.0,232485.33125,0.958800,360.3125,0.0
3,1.5,223547.53125,0.973093,235.3125,0.0
4,2.0,218410.70625,0.983927,140.5625,0.0
5,2.5,217488.20000,0.990917,79.4375,0.0
6,3.0,220195.64375,0.994626,47.0000,0.0


,sku,week,order_qty
0,FOODS_3_064_CA_1_validation,0,105
1,FOODS_3_090_CA_1_validation,0,417
2,FOODS_3_120_CA_1_validation,0,302
3,FOODS_3_252_CA_1_validation,0,141
4,FOODS_3_586_CA_1_validation,0,105
5,FOODS_3_587_CA_1_validation,0,126
6,FOODS_3_064_CA_1_validation,1,200
7,FOODS_3_090_CA_1_validation,1,354
8,FOODS_3_120_CA_1_validation,1,218
9,FOODS_3_252_CA_1_validation,1,231


## Save outputs

In [7]:
forecast.to_csv(DATA_DIR/'forecast.csv',index=False)
train_scenarios.to_csv(DATA_DIR/'train_scenarios.csv',index=False)
params.reset_index().to_csv(DATA_DIR/'sku_parameters_ASSUMED.csv',index=False)
(DATA_DIR/'config.json').write_text(json.dumps(conf,indent=2)+'\n',encoding='utf-8')
forecast_check.to_csv(DATA_DIR/'forecast_holdout_errors.csv',index=False)
print('Saved forecast, train_scenarios, assumed SKU parameters, config and forecast errors.')

# One common baseline plan is shared by BOTH solver notebooks; the plan uses no holdout data.

baseline_orders.to_csv(DATA_DIR/'baseline_orders.csv', index=False)
baseline_tuning.to_csv(DATA_DIR/'baseline_training_tuning.csv', index=False)
(DATA_DIR/'baseline_settings.json').write_text(json.dumps({
    'policy': 'fixed forecast-driven base-stock-style schedule',
    'selected_training_buffer_multiplier': BASELINE_BUFFER,
    'training_service_target_met': bool(baseline_training_service_met),
    'training_fill_rate': float(selected['train_fill_rate']),
    'training_cost': float(selected['train_cost']),
    'tuning_uses_holdout': False,
}, indent=2) + '\n', encoding='utf-8')
print('Saved common baseline_orders.csv and baseline_training_tuning.csv (training only).')

Saved forecast, train_scenarios, assumed SKU parameters, config and forecast errors.
Saved common baseline_orders.csv and baseline_training_tuning.csv (training only).
